# Lucida RVC Server — Colab Setup

**Purpose:** Run RVC inference on GPU (Colab T4), expose as Gradio `/infer` endpoint  
**Consumer:** `rvc_agent.py` on your local Surface Laptop 2  

### Steps
1. Run **Cell 1** — install deps  
2. Run **Cell 2** — upload your voice model (`.pth`) + index (`.index`)  
3. Run **Cell 3** — start server, copy the public URL  
4. Paste URL into `.env` as `GRADIO_API_URL=`

In [ ]:
# ── Cell 1: Install ───────────────────────────────────────────────────────────
!pip install -q gradio==4.44.1 librosa soundfile praat-parselmouth faiss-cpu
!pip install -q torch torchcrepe torchfcpe

# Clone minimal RVC inference (no full WebUI — just inference core)
import os
if not os.path.exists('rvc-inferpy'):
    !git clone -q https://github.com/fumiama/RVC-inference rvc-inferpy

%cd rvc-inferpy
!pip install -q -r requirements.txt

# Download pretrained models needed for rmvpe
import subprocess
os.makedirs('assets/rmvpe', exist_ok=True)
os.makedirs('assets/hubert', exist_ok=True)

RMVPE_URL = 'https://huggingface.co/lj1995/VoiceConversionWebUI/resolve/main/rmvpe.pt'
HUBERT_URL = 'https://huggingface.co/lj1995/VoiceConversionWebUI/resolve/main/hubert_base.pt'

!wget -q -nc -O assets/rmvpe/rmvpe.pt {RMVPE_URL}
!wget -q -nc -O assets/hubert/hubert_base.pt {HUBERT_URL}

print('✓ Install complete')

In [ ]:
# ── Cell 2: Upload voice model ────────────────────────────────────────────────
from google.colab import files
import shutil, os

os.makedirs('assets/weights', exist_ok=True)
os.makedirs('assets/index', exist_ok=True)

print('Upload your .pth model file:')
uploaded = files.upload()
for fname in uploaded:
    if fname.endswith('.pth'):
        shutil.move(fname, f'assets/weights/{fname}')
        MODEL_PATH = f'assets/weights/{fname}'
        print(f'  Model: {MODEL_PATH}')

print('\nUpload your .index file (or skip if you have none):')
uploaded2 = files.upload()
INDEX_PATH = ''
for fname in uploaded2:
    if fname.endswith('.index'):
        shutil.move(fname, f'assets/index/{fname}')
        INDEX_PATH = f'assets/index/{fname}'
        print(f'  Index: {INDEX_PATH}')

if not uploaded2:
    print('  (no index — index_rate will be ignored)')

In [ ]:
# ── Cell 3: Start Gradio RVC server ──────────────────────────────────────────
import gradio as gr
import torch, os, sys
sys.path.insert(0, '.')

# ── RVC inference wrapper ─────────────────────────────────────────────────────
from configs.config import Config
from infer.modules.vc.modules import VC

config = Config()
vc = VC(config)
vc.get_vc(MODEL_PATH)

def infer(
    input_audio,       # filepath from Gradio
    f0_up_key,         # pitch shift — MUST be 0 from rvc_agent
    f0_method,         # rmvpe
    index_file,        # index filepath or empty
    index_rate,
    filter_radius,
    resample_sr,
    rms_mix_rate,
    protect,
):
    idx = INDEX_PATH if (INDEX_PATH and index_rate > 0) else ''
    _, wav_opt = vc.vc_single(
        sid=0,
        input_audio_path=input_audio,
        f0_up_key=int(f0_up_key),
        f0_file=None,
        f0_method=f0_method,
        file_index=idx,
        file_index2='',
        index_rate=float(index_rate),
        filter_radius=int(filter_radius),
        resample_sr=int(resample_sr),
        rms_mix_rate=float(rms_mix_rate),
        protect=float(protect),
    )
    # wav_opt is (sample_rate, numpy_array)
    import soundfile as sf, tempfile
    tmp = tempfile.mktemp(suffix='.wav')
    sf.write(tmp, wav_opt[1], wav_opt[0])
    return tmp

# ── Gradio interface ──────────────────────────────────────────────────────────
with gr.Blocks() as demo:
    gr.Markdown(f'## Lucida RVC Server — model: `{os.path.basename(MODEL_PATH)}`')
    with gr.Row():
        inp   = gr.Audio(type='filepath', label='Input audio')
        out   = gr.Audio(type='filepath', label='Converted audio')
    f0_key   = gr.Number(value=0,    label='Pitch shift (0 = preserve)')
    f0_m     = gr.Textbox(value='rmvpe', label='F0 method')
    idx_file = gr.Textbox(value='',  label='Index file (auto)')
    idx_rate = gr.Number(value=0.75, label='Index rate')
    filt_r   = gr.Number(value=3,    label='Filter radius')
    resamp   = gr.Number(value=0,    label='Resample SR')
    rms      = gr.Number(value=0.25, label='RMS mix rate')
    prot     = gr.Number(value=0.33, label='Protect')
    btn = gr.Button('Convert')
    btn.click(
        fn=infer,
        inputs=[inp, f0_key, f0_m, idx_file, idx_rate, filt_r, resamp, rms, prot],
        outputs=out,
        api_name='infer',
    )

demo.launch(share=True, quiet=True)
# ↑ Outputs a public URL like: https://xxxx.gradio.live
# Copy that URL into .env as GRADIO_API_URL=

## Sau khi có URL

Copy URL từ output của Cell 3, paste vào `.env` trên máy local:

```
GRADIO_API_URL=https://xxxx.gradio.live
```

Rồi chạy pipeline:

```powershell
cd "C:\Users\HUY\AI\OPUS ANIMUS\opus-lucida"
python automation/video/pipeline.py wake-cluster --skip-screenshot
```

Pipeline sẽ:
1. Gen TTS audio (edge-tts + Voicevox)
2. Pass từng slide qua RVC trên Colab T4
3. Assembly slides + converted audio → MP4

### Lưu ý
- Colab free T4 session giới hạn ~4h — nếu hết session, chạy lại Cell 3
- `--skip-tts` để reuse audio đã gen, chỉ chạy lại RVC
- Nếu `GRADIO_API_URL` trống, pipeline tự skip RVC và dùng raw TTS audio